In [3]:
"""
This file builds the RAG system.

Steps happening here:
1. Load policy document
2. Split into chunks
3. Convert chunks into embeddings
4. Store embeddings in vector database
5. Create retriever function to query policies
"""

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [4]:
# -----------------------------------------
# Load loan policy document
# -----------------------------------------

loader = TextLoader("data/loan_rules.txt")
documents = loader.load()

In [5]:
# -----------------------------------------
# Split document into chunks
# -----------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

docs = splitter.split_documents(documents)

In [6]:
# -----------------------------------------
# Convert chunks into embeddings
# -----------------------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# -----------------------------------------
# Store embeddings into vector DB
# -----------------------------------------

vectorstore = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory="loan_policy_db"
)

vectorstore.persist()

C:\Users\User\AppData\Local\Temp\ipykernel_7660\4258629191.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\User\AppData\Local\Temp\ipykernel_7660\4258629191.py:20: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [10]:
# -----------------------------------------
# Retriever function used by agents
# -----------------------------------------

def query_policies(question: str) -> str:
    """
    This function retrieves relevant policy rules from the vector DB.

    Input:
        question -> what we want to know about policies

    Output:
        concatenated policy rules text
    """

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    docs = retriever.invoke(question)

    return "\n".join([d.page_content for d in docs])

In [11]:

# quick test
if __name__ == "__main__":
    result = query_policies("loan eligibility rules")
    print(result)

Loan Policy Rules

1. Minimum salary required for personal loan is 25,000 per month.

2. Minimum credit score required is 700.

3. Maximum Debt To Income (DTI) ratio allowed is 55%.
4. Maximum loan amount allowed is 20 times the monthly salary.

5. Interest rate typically ranges between 9% to 13%.

6. Maximum loan tenure allowed is 7 years.
7. Existing EMI obligations should not exceed 40% of salary.

8. Applicants with credit score above 750 get priority approval.

9. Applicants with credit score below 650 are considered high risk.


In [28]:
"""
Agentic Loan Eligibility Assistant using LangGraph + RAG

Workflow:

User Input
   ↓
collect_financials
   ↓
check_eligibility
   ↓
predict_approval_chance
   ↓
suggest_loan_plan
   ↓
review_response
   ↓
END
"""

import re
from typing import Optional

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

# Updated LangChain imports
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_community.chat_models import ChatOllama

# RAG retriever
# from build_rag import query_policies


# =====================================================
# 1️⃣ State Definition
# =====================================================

class LoanState(BaseModel):
    """
    This class stores the information flowing through the graph.

    LangGraph passes this state between nodes.
    Each node can read/update values.
    """

    user_input: str

    salary: Optional[int] = Field(default=None)
    credit_score: Optional[int] = Field(default=None)
    loan_amount: Optional[int] = Field(default=None)
    tenure_years: Optional[int] = Field(default=None)
    existing_emi: Optional[int] = Field(default=None)

    retrieved_rules: Optional[str] = None
    eligibility_status: Optional[str] = None
    approval_score: Optional[float] = None
    suggested_plan: Optional[str] = None
    final_reply: Optional[str] = None


# =====================================================
# 2️⃣ Node 1 — Extract financial details
# =====================================================

def collect_financials(state: LoanState):
    """
    Extracts salary, credit score, loan amount, tenure, EMI
    from user text using regex.
    """

    text = state.user_input.lower()

    # extract salary
    salary_match = re.search(r'(\d+)\s?k', text)
    if salary_match:
        state.salary = int(salary_match.group(1)) * 1000

    # extract credit score
    credit_match = re.search(r'credit score\s*(\d+)', text)
    if credit_match:
        state.credit_score = int(credit_match.group(1))

    # extract loan amount
    loan_match = re.search(r'(\d+)\s?lakhs?', text)
    if loan_match:
        state.loan_amount = int(loan_match.group(1)) * 100000

    # extract tenure
    tenure_match = re.search(r'(\d+)\s?yrs?', text)
    if tenure_match:
        state.tenure_years = int(tenure_match.group(1))

    # extract existing EMI
    emi_match = re.search(r'emi\s*(\d+)k', text)
    if emi_match:
        state.existing_emi = int(emi_match.group(1)) * 1000

    return state


# =====================================================
# 3️⃣ Node 2 — Eligibility Check using RAG
# =====================================================

def check_eligibility(state: LoanState):
    """
    Uses RAG + LLM reasoning to determine loan eligibility.

    Steps happening here:
    1. Retrieve bank policies using RAG
    2. Send policies + user financials to LLM
    3. LLM decides eligibility dynamically
    """

    # ------------------------------------------------
    # Step 1: Retrieve policies using RAG
    # ------------------------------------------------

    rules = query_policies("loan eligibility rules and requirements")
    state.retrieved_rules = rules

    # ------------------------------------------------
    # Step 2: Define structured output format
    # ------------------------------------------------

    response_schemas = [
        ResponseSchema(name="eligibility", description="Eligible or Not Eligible"),
        ResponseSchema(name="reason", description="Short explanation"),
    ]

    parser = StructuredOutputParser.from_response_schemas(response_schemas)

    format_instructions = parser.get_format_instructions()

    # ------------------------------------------------
    # Step 3: Create reasoning prompt
    # ------------------------------------------------

    prompt = PromptTemplate(
        template="""
You are a banking loan approval assistant.

Use the bank policies to determine if the user qualifies.

Bank Policies:
{rules}

User Financial Profile:
Salary: {salary}
Credit Score: {credit_score}
Loan Amount: {loan_amount}
Tenure: {tenure}
Existing EMI: {emi}

Determine loan eligibility strictly based on policies.

{format_instructions}
""",
        input_variables=[
            "rules",
            "salary",
            "credit_score",
            "loan_amount",
            "tenure",
            "emi"
        ],
        partial_variables={"format_instructions": format_instructions},
    )

    # ------------------------------------------------
    # Step 4: Send reasoning request to LLM
    # ------------------------------------------------

    chain = prompt | llm

    response = chain.invoke({
        "rules": rules,
        "salary": state.salary,
        "credit_score": state.credit_score,
        "loan_amount": state.loan_amount,
        "tenure": state.tenure_years,
        "emi": state.existing_emi
    })

    # ------------------------------------------------
    # Step 5: Parse structured response
    # ------------------------------------------------

    parsed = parser.parse(response.content)

    state.eligibility_status = parsed["eligibility"]

    return state


# =====================================================
# 4️⃣ Node 3 — Predict approval chance
# =====================================================

def predict_approval_chance(state: LoanState):
    """
    Simple heuristic scoring system.

    Factors used:
    - credit score
    - salary
    - DTI
    """

    score = 50

    if state.credit_score:
        score += (state.credit_score - 650) * 0.2

    if state.salary and state.loan_amount:
        loan_ratio = state.loan_amount / (state.salary * 12)

        if loan_ratio < 5:
            score += 10
        elif loan_ratio > 10:
            score -= 10

    if state.salary and state.existing_emi:
        dti = state.existing_emi / state.salary

        if dti < 0.3:
            score += 10
        else:
            score -= 10

    score = max(0, min(100, score))

    state.approval_score = round(score, 2)

    return state


# =====================================================
# 5️⃣ Node 4 — Suggest loan plan
# =====================================================

def suggest_loan_plan(state: LoanState):
    """
    Suggests EMI plan.

    EMI formula used:
    EMI = P * r * (1+r)^n / ((1+r)^n - 1)
    """

    if not state.loan_amount or not state.tenure_years:
        return state

    P = state.loan_amount

    annual_rate = 0.10
    r = annual_rate / 12

    n = state.tenure_years * 12

    emi = (P * r * (1 + r)**n) / ((1 + r)**n - 1)

    emi = int(emi)

    state.suggested_plan = f"""
Loan Amount: ₹{P:,}
Tenure: {state.tenure_years} years
Estimated EMI: ₹{emi:,}
Interest Assumption: 10%
"""

    return state


# =====================================================
# 6️⃣ Node 5 — Final response formatting
# =====================================================

def review_response(state: LoanState):
    """
    Formats the final clean answer for the user.
    """

    state.final_reply = f"""
Loan Eligibility Result
------------------------

Eligibility Status: {state.eligibility_status}

Approval Probability: {state.approval_score}%

Recommended Loan Plan:
{state.suggested_plan}

Advice:
Maintain a strong credit score and keep EMI obligations low.
"""

    return state


# =====================================================
# 7️⃣ Build LangGraph Workflow
# =====================================================

workflow = StateGraph(LoanState)

workflow.add_node("collect_financials", collect_financials)
workflow.add_node("check_eligibility", check_eligibility)
workflow.add_node("predict_approval_chance", predict_approval_chance)
workflow.add_node("suggest_loan_plan", suggest_loan_plan)
workflow.add_node("review_response", review_response)


workflow.add_edge("collect_financials", "check_eligibility")
workflow.add_edge("check_eligibility", "predict_approval_chance")
workflow.add_edge("predict_approval_chance", "suggest_loan_plan")
workflow.add_edge("suggest_loan_plan", "review_response")
workflow.add_edge("review_response", END)


workflow.set_entry_point("collect_financials")

loan_app = workflow.compile()



ImportError: cannot import name 'StructuredOutputParser' from 'langchain_core.output_parsers' (C:\Users\User\.conda\envs\training_env\lib\site-packages\langchain_core\output_parsers\__init__.py)

In [22]:

# =====================================================
# 8️⃣ Example Run
# =====================================================

if __name__ == "__main__":

    query = "I earn 65k/month, credit score 740, loan required 10 lakhs for 3 yrs. Already paying emi 12k."

    result = loan_app.invoke({"user_input": query})

    print(result["final_reply"])

NameError: name 'ResponseSchema' is not defined

In [16]:
!pip install fastapi-responseschema



  Using cached fastapi-0.135.1-py3-none-any.whl.metadata (30 kB)
  Using cached starlette-0.52.1-py3-none-any.whl.metadata (6.3 kB)
Using cached fastapi-0.135.1-py3-none-any.whl (116 kB)
Using cached starlette-0.52.1-py3-none-any.whl (74 kB)

   ---------------------------------------- 0/3 [starlette]
   ---------------------------------------- 0/3 [starlette]
   ---------------------------------------- 0/3 [starlette]
   ---------------------------------------- 0/3 [starlette]
   ---------------------------------------- 0/3 [starlette]
   ------------- -------------------------- 1/3 [fastapi]
   ------------- -------------------------- 1/3 [fastapi]
   ------------- -------------------------- 1/3 [fastapi]
   ------------- -------------------------- 1/3 [fastapi]
   ------------- -------------------------- 1/3 [fastapi]
   ------------- -------------------------- 1/3 [fastapi]
   ------------- -------------------------- 1/3 [fastapi]
   ---------------------------------------- 3/3 [fa